# GPU runs on Colab
Runs the GPU stage (`cell_mlp`, `mil_attn`, optional extras). Uses Google Drive so the derived data and results survive Colab timeouts: every (model, seed, fold) is cached, so re-running a cell resumes.

**Before this:** build the derived data locally (`psychad build`) and upload `data/derived/full/` to Drive at `MyDrive/psychad/data/derived/full/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi | head -15

In [ ]:
# replace with your GitHub repo URL once pushed
REPO = 'https://github.com/<user>/<repo>.git'
!git clone $REPO /content/proj || (cd /content/proj && git pull)
%cd /content/proj
!pip install -q -e '.[torch,xgb]'

In [ ]:
import os
# point data/ and results/ at Drive so they persist (symlinks)
os.makedirs('/content/drive/MyDrive/psychad/data', exist_ok=True)
os.makedirs('/content/drive/MyDrive/psychad/results', exist_ok=True)
!ln -sfn /content/drive/MyDrive/psychad/data /content/proj/data
!ln -sfn /content/drive/MyDrive/psychad/results /content/proj/results
!ls data/derived/full | head

In [ ]:
# sanity check on the GPU with the fast profile first (time this!)
!python -m psychad_ml --profile dev benchmark --task four_class --models cell_mlp

In [ ]:
# real run (resumable: just re-run this cell after a timeout)
!python -m psychad_ml benchmark --task four_class lbd_group --models cell_mlp mil_attn

In [ ]:
!python -m psychad_ml celltype --task four_class --models cell_mlp
!python -m psychad_ml report --task four_class

In [ ]:
# optional extras
# !python -m psychad_ml benchmark --task four_class --models cell_mlp_large
# !python -m psychad_ml scaling --task four_class